In [2]:
import numpy as np
import pandas as pd

| Table | Why we need it |
|---|---|
| `CaseMaster.csv` | The fact table — one row per FIR, with station, crime type, date, coordinates, gravity |
| `GravityOffence.csv` | Lookup to turn `GravityOffenceID` into `Heinous` / `Serious` / etc., so we can compute `heinous_count` |
| `ChargesheetDetails.csv` | Tells us how each case was finally resolved (`cstype`), which feeds `resolution_rate` |

Everything else in the 26 files (Employee, Unit, Rank, District...) is
relevant to *other* modules or to enrichment, but not to this aggregation.

In [4]:
cm = pd.read_csv("../cleaned_data/Cleaned_CaseMaster.csv")
gravity = pd.read_csv("../synthetic_data/GravityOffence.csv")
cs = pd.read_csv("../cleaned_data/chargesheet_cleaned.csv")

In [14]:
print("CaseMaster shape:", cm.shape)
print("GravityOffence shape:", gravity.shape)
print("ChargesheetDetails shape:", cs.shape)

CaseMaster shape: (15000, 18)
GravityOffence shape: (4, 2)
ChargesheetDetails shape: (13228, 5)


### Confirming if:
- there are no nulls in the columns that are going to be grouped
- the ID's that are going to be aggregated aren't some wierd mix (for example if range offices are mixed with actual police stations)

In [15]:
print("Unique Stations: ",cm["PoliceStationID"].nunique())
print("Unique crime sub-heads: ",cm["CrimeMinorHeadID"].nunique())
print("Nulls in key columns: ")
print(cm[["PoliceStationID","CrimeMinorHeadID","GravityOffenceID","IncidentFromDate"]].isnull().sum())

Unique Stations:  129
Unique crime sub-heads:  63
Nulls in key columns: 
PoliceStationID     0
CrimeMinorHeadID    0
GravityOffenceID    0
IncidentFromDate    0
dtype: int64


Deriving `period` from IncidentFromDate (when crime actually happened). The period will be monthly because it gives a denser and more learnable signal, at the cose of shorter forecast horizons.

In [16]:
cm.info()

<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CaseMasterID         15000 non-null  int64  
 1   CrimeNo              15000 non-null  int64  
 2   CaseNo               15000 non-null  int64  
 3   CrimeRegisteredDate  15000 non-null  str    
 4   PolicePersonID       15000 non-null  int64  
 5   PoliceStationID      15000 non-null  int64  
 6   CaseCategoryID       15000 non-null  int64  
 7   GravityOffenceID     15000 non-null  int64  
 8   CrimeMajorHeadID     15000 non-null  int64  
 9   CrimeMinorHeadID     15000 non-null  int64  
 10  CaseStatusID         15000 non-null  int64  
 11  CourtID              15000 non-null  int64  
 12  IncidentFromDate     15000 non-null  str    
 13  IncidentToDate       15000 non-null  str    
 14  InfoReceivedPSDate   15000 non-null  str    
 15  latitude             15000 non-null  float64
 1

In [6]:
cm["CrimeRegisteredDate"] = pd.to_datetime(cm["CrimeRegisteredDate"], errors='coerce')
cm["IncidentFromDate"] = pd.to_datetime(cm["IncidentFromDate"], errors = 'coerce')
cm["IncidentToDate"] = pd.to_datetime(cm["IncidentToDate"], errors= 'coerce')
cm["InfoReceivedPSDate"] = pd.to_datetime(cm["InfoReceivedPSDate"], errors = 'coerce')

In [7]:
cm["period"] = cm["IncidentFromDate"].values.astype("datetime64[M]")
cm[['IncidentFromDate','period']].head()

,IncidentFromDate,period
0,2025-11-20 10:07:25,2025-11-01
1,2025-12-16 16:21:47,2025-12-01
2,2023-10-28 17:15:04,2023-10-01
3,2025-02-22 07:40:32,2025-02-01
4,2022-11-15 08:19:30,2022-11-01


`GravityOffenceID` is a foreign key into GravityOffence. We join it in, then derive a simple 0/1 flag we can later sum per to get `heinous_count`

In [8]:
cm = cm.merge(gravity, on = "GravityOffenceID", how = "left")
cm["is_heinous"] = (cm["LookupValue"] == "Heinous").astype(int)

cm[["GravityOffenceID","LookupValue","is_heinous"]].drop_duplicates()

,GravityOffenceID,LookupValue,is_heinous
0,2,Serious,0
2,3,Non-Heinous,0
8,1,Heinous,1
17,4,Petty,0


It is observed that not every case has a `ChargesheetDetails` row yet - some are still under investigation. If we just did an inner join, those cases would silently vanish from our counts. We need a left join, and we need to explicitly decide what "no chargesheet yet" means (here: not resolved)
We also take the latest chargesheet row per case, in case a case has more than one row (defensive, in case of data entry corrections).

In [9]:
cs_latest = cs.sort_values("csdate").groupby("CaseMasterID", as_index=False).last()

cm = cm.merge(cs_latest[["CaseMasterID", "cstype"]], on="CaseMasterID", how="left")

n_no_chargesheet = cm["cstype"].isnull().sum()
print(f"Cases with no chargesheet row yet: {n_no_chargesheet} (treated as unresolved)")

cm["is_resolved"] = (cm["cstype"] == "A").astype(int)

Cases with no chargesheet row yet: 1772 (treated as unresolved)


If we aggregate first and only keep combinations that actually had cases,
we never get the zero rows — there's nothing to "fill," the row simply
never existed. So we build the **complete cartesian product** first:

In [10]:
all_stations = cm["PoliceStationID"].unique()
all_crimes = cm["CrimeMinorHeadID"].unique()
full_range = pd.date_range(cm["period"].min(), cm["period"].max(), freq="MS")

scaffold = pd.MultiIndex.from_product(
    [all_stations, all_crimes, full_range],
    names=["PoliceStationID", "CrimeMinorHeadID", "period"]
).to_frame(index=False)

print(f"Scaffold size: {len(scaffold):,} rows "
      f"({len(all_stations)} stations x {len(all_crimes)} crime types x {len(full_range)} months)")
scaffold.head()

Scaffold size: 552,636 rows (129 stations x 63 crime types x 68 months)


,PoliceStationID,CrimeMinorHeadID,period
0,83,30,2020-12-01
1,83,30,2021-01-01
2,83,30,2021-02-01
3,83,30,2021-03-01
4,83,30,2021-04-01


In [11]:
agg = cm.groupby(["PoliceStationID", "CrimeMinorHeadID", "period"]).agg(
    case_count=("CaseMasterID", "count"),
    heinous_count=("is_heinous", "sum"),
    resolved_count=("is_resolved", "sum"),
).reset_index()

agg.head()

,PoliceStationID,CrimeMinorHeadID,period,case_count,heinous_count,resolved_count
0,73,1,2021-05-01,1,1,1
1,73,1,2023-11-01,1,1,1
2,73,1,2024-03-01,1,1,1
3,73,1,2024-05-01,1,1,0
4,73,1,2024-07-01,1,1,0


- `case_count` — how many FIRs
- `heinous_count` — sum of the heinous flag
- `resolved_count` — sum of the resolved flag (we'll turn this into a rate next)

`resolution_rate`:** when `case_count == 0`, "resolved / total"
is undefined (0/0). We set it to `0.0` for simplicity, but this is a
*different meaning* from "cases happened, none were resolved" — worth a
comment/flag if this feeds a dashboard tooltip later.


In [12]:
df = scaffold.merge(agg, on=["PoliceStationID", "CrimeMinorHeadID", "period"], how="left")

df["case_count"] = df["case_count"].fillna(0).astype(int)
df["heinous_count"] = df["heinous_count"].fillna(0).astype(int)
df["resolved_count"] = df["resolved_count"].fillna(0).astype(int)

df["resolution_rate"] = np.where(
    df["case_count"] > 0, df["resolved_count"] / df["case_count"], 0.0
)

df = df.drop(columns=["resolved_count"])
df = df.sort_values(["PoliceStationID", "CrimeMinorHeadID", "period"]).reset_index(drop=True)

df.head(10)

,PoliceStationID,CrimeMinorHeadID,period,case_count,heinous_count,resolution_rate
0,73,1,2020-12-01,0,0,0.0
1,73,1,2021-01-01,0,0,0.0
2,73,1,2021-02-01,0,0,0.0
3,73,1,2021-03-01,0,0,0.0
4,73,1,2021-04-01,0,0,0.0
5,73,1,2021-05-01,1,1,1.0
6,73,1,2021-06-01,0,0,0.0
7,73,1,2021-07-01,0,0,0.0
8,73,1,2021-08-01,0,0,0.0
9,73,1,2021-09-01,0,0,0.0


Three checks that would catch the most common mistakes in this kind of
aggregation:

1. The scaffold has exactly `stations x crimes x months` rows — no more, no less.
2. The total of `case_count` across the whole table equals the number of
   raw rows in `CaseMaster` — i.e. we haven't lost or duplicated any cases.
3. No nulls remain in any of the core columns.

In [13]:
n_expected = len(all_stations) * len(all_crimes) * len(full_range)
assert len(df) == n_expected, f"scaffold mismatch: {len(df)} vs {n_expected}"

assert df["case_count"].sum() == cm.shape[0], "case_count total doesn't match raw CaseMaster rows"

core_cols = ["PoliceStationID", "CrimeMinorHeadID", "period", "case_count", "heinous_count", "resolution_rate"]
assert df[core_cols].isnull().sum().sum() == 0, "unexpected nulls in core columns"

print("All checks passed.")
print(f"Total cases in scaffold: {int(df['case_count'].sum()):,} | raw CaseMaster rows: {cm.shape[0]:,}")
print(f"Rows with case_count > 0: {(df['case_count'] > 0).sum():,} ({(df['case_count'] > 0).mean():.1%} of all rows)")


All checks passed.
Total cases in scaffold: 15,000 | raw CaseMaster rows: 15,000
Rows with case_count > 0: 14,435 (2.6% of all rows)


Prophet and ARIMA fit one series at a time and don't need this section.
If instead you want a **single cross-station model**, forecasting becomes
a supervised regression problem, and needs explicit lag/seasonal features.

lag and rolling features must be computed *within each
`(PoliceStationID, CrimeMinorHeadID)` group*, sorted by `period` — otherwise
one station's history leaks into another station's row.

Note that early rows in each group will have `NaN` lags (there's no
history yet for month 1, 2, 3...) — that's correct, not a bug. Drop or
impute them later; don't zero-fill, since `0` here would look like "no
crime last month" instead of "no data yet.

In [14]:
df["month"] = df["period"].dt.month
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

g = df.groupby(["PoliceStationID", "CrimeMinorHeadID"])["case_count"]
df["count_lag_1"] = g.shift(1)
df["count_lag_3"] = g.shift(3)
df["count_lag_12"] = g.shift(12)
df["rolling_mean_3"] = g.transform(lambda s: s.shift(1).rolling(3).mean())

df.head(10)

,PoliceStationID,CrimeMinorHeadID,period,case_count,heinous_count,resolution_rate,month,month_sin,month_cos,count_lag_1,count_lag_3,count_lag_12,rolling_mean_3
0,73,1,2020-12-01,0,0,0.0,12,-2.449294e-16,1.000000e+00,NaN,NaN,NaN,NaN
1,73,1,2021-01-01,0,0,0.0,1,5.000000e-01,8.660254e-01,0.0,NaN,NaN,NaN
2,73,1,2021-02-01,0,0,0.0,2,8.660254e-01,5.000000e-01,0.0,NaN,NaN,NaN
3,73,1,2021-03-01,0,0,0.0,3,1.000000e+00,6.123234e-17,0.0,0.0,NaN,0.000000
4,73,1,2021-04-01,0,0,0.0,4,8.660254e-01,-5.000000e-01,0.0,0.0,NaN,0.000000
5,73,1,2021-05-01,1,1,1.0,5,5.000000e-01,-8.660254e-01,0.0,0.0,NaN,0.000000
6,73,1,2021-06-01,0,0,0.0,6,1.224647e-16,-1.000000e+00,1.0,0.0,NaN,0.333333
7,73,1,2021-07-01,0,0,0.0,7,-5.000000e-01,-8.660254e-01,0.0,0.0,NaN,0.333333
8,73,1,2021-08-01,0,0,0.0,8,-8.660254e-01,-5.000000e-01,0.0,1.0,NaN,0.333333
9,73,1,2021-09-01,0,0,0.0,9,-1.000000e+00,-1.836970e-16,0.0,0.0,NaN,0.000000


For any of Prophet, ARIMA, or LightGBM, a **random row split leaks the
future into training**. The correct split holds out the last N months
*per series* as test data.

In [15]:
N_TEST_MONTHS = 6
cutoff = df["period"].max() - pd.DateOffset(months=N_TEST_MONTHS)

train = df[df["period"] <= cutoff]
test = df[df["period"] > cutoff]

print(f"Cutoff: {cutoff.date()}")
print(f"Train rows: {len(train):,} | Test rows: {len(test):,}")

Cutoff: 2026-01-01
Train rows: 503,874 | Test rows: 48,762


Two files:
- `timeseries_counts.csv` — the exact spec format (core columns only), for
  Prophet/ARIMA/graders expecting the plain spec.
- `timeseries_counts_supervised.csv` — the extended version with lag and
  seasonal features, for a LightGBM/XGBoost approach.

In [16]:
df[core_cols].to_csv("timeseries_counts.csv", index=False)
df.to_csv("timeseries_counts_supervised.csv", index=False)

print("Saved timeseries_counts.csv and timeseries_counts_supervised.csv")

Saved timeseries_counts.csv and timeseries_counts_supervised.csv


1. **Missing rows ≠ zeros.** Built the full cartesian scaffold before joining real counts, so every station×crime×month combo exists explicitly.
2. **Silent case loss.** Used a left join for chargesheet status so cases still under investigation don't vanish from counts.
3. **0/0 ambiguity.** `resolution_rate` on a zero-case month is set to `0.0` but is *not* the same claim as "cases existed and none were resolved."
4. **Lag leakage across groups.** All lag/rolling features are computed within `(PoliceStationID, CrimeMinorHeadID)` groups, sorted by time, and split train/test by date — never randomly.